## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Six social platform types: a map from similarity ratings

**Hayashi's Quantification Method IV (Matrix Format)** · a worked example from the StatNex teaching guide

**The story.** Users rated how similar six types of social platform feel to them (1 = very different, 5 = very similar), for every pair. An advertiser wants to know which platform types users see as alike.

**The question.** Which platform types do users see as alike, and so as possible substitutes for an advertiser, and which stand apart?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/quantification_iv_matrix.csv` (6 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Platform,Photo sharing,Short video,Microblog,Messaging,Professional network,Video streaming
Photo sharing,,,,,,
Short video,5.0,,,,,
Microblog,3.0,3.0,,,,
Messaging,2.0,2.0,3.0,,,
Professional network,1.0,1.0,3.0,2.0,,
Video streaming,3.0,5.0,2.0,1.0,1.0,
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The matrix

In [ ]:
# A square table: one row and one column per object. Only one triangle need be
# typed; when the other is blank it is filled in by symmetry, and a blank diagonal
# is 0.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Photo sharing", "Short video", "Microblog", "Messaging", "Professional network", "Video streaming"]
label_column = ["Platform"]
matrix = data[columns].apply(pd.to_numeric, errors="coerce").to_numpy(float)
labels = data[label_column[0]].astype(str).tolist() if label_column else columns
n = len(matrix)
upper_blank = np.isnan(matrix[np.triu_indices(n, 1)]).all()
lower_blank = np.isnan(matrix[np.tril_indices(n, -1)]).all()
matrix = np.nan_to_num(matrix, nan=0.0)
if upper_blank or lower_blank:
    matrix = matrix + matrix.T - np.diag(np.diag(matrix))
print(pd.DataFrame(matrix, index=labels, columns=labels))

# The entries are similarities: large means "belongs together".
similarity = matrix

## H matrix

In [ ]:
# Hayashi's method gives each object a score x so that similar objects get close
# scores: it maximises -sum over pairs of (e_ij + e_ji)(x_i - x_j)^2 with the scores
# centred and their squares summing to 1. That sum is x'Hx for this H: off the
# diagonal, e_ij + e_ji; on it, minus the rest of the row, so each row sums to 0.
# Read the diagonal: it is minus twice an object's total similarity to all the
# others. An object with little similarity to anything (a small diagonal) costs
# least to move away from the rest, so it tends to get an axis of its own.
h = similarity + similarity.T
np.fill_diagonal(h, 0.0)
np.fill_diagonal(h, -h.sum(axis=1))
print()
print(pd.DataFrame(h, index=labels, columns=labels).round(4))

## Eigenvalues and eigenvectors

In [ ]:
# Each eigenvalue is the value of the sum above for its dimension's scores. With
# positive similarities every one is negative, and the closer to 0, the better that
# dimension keeps similar objects together; the first dimension is the best. There
# is no "share explained" here, unlike in MDS.
# Because every row sums to 0, the constant vector is always an eigenvector, with
# eigenvalue 0: "put everything in one place", which says nothing. So the search is
# made among vectors orthogonal to it: the columns of `basis` span exactly those,
# H is solved inside that space, and the result is mapped back. Largest first, each
# turned to make its first element positive (a map and its mirror image are the
# same).
basis = np.linalg.qr(np.column_stack([np.ones(n), np.eye(n)[:, 1:]]))[0][:, 1:]
reduced = basis.T @ h @ basis
eigenvalues, inner = np.linalg.eigh((reduced + reduced.T) / 2)
order = np.argsort(-eigenvalues)
eigenvalues = eigenvalues[order]
vectors = basis @ inner[:, order]
vectors = vectors * np.where(vectors[0] < 0, -1.0, 1.0)
dims = [f"Dim {j + 1}" for j in range(n - 1)]
print()
print(pd.Series(eigenvalues, index=dims))
print()
print(pd.DataFrame(vectors, index=labels, columns=dims).round(4))

## The map

In [ ]:
# The first eigenvectors, as they are: unit-length scores with no units, so the map
# keeps which objects are near which, not how similar they were rated.
dimensions = min(2, n - 1)
projection = vectors[:, :dimensions]
print()
print(pd.DataFrame(projection, index=labels, columns=dims[:dimensions]).round(4))

## What the result says

Quantification method IV draws a map from a table of similarities: pairs that users rate as alike are pulled together. Here six platform types were rated in pairs from 1 (very different) to 5 (very similar).

The ratings already show the core: short video scores 5 with both photo sharing and video streaming. On dimension 1 of the projection these three sit together on the positive side (0.30, 0.28 and 0.40), and the professional network is far away at -0.80. Dimension 2 separates messaging, alone at -0.84. Microblog sits near the centre (-0.05 and -0.06), close to no family.

So users see video streaming, short video and photo sharing as one family, and so as possible substitutes for an advertiser. The professional network and messaging each stand apart. The map shows how users perceive the platforms, not where they spend their time.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.